# 02 — Arabic Text Cleaning

**Pipeline:** `00_setup.ipynb` → **`02_text_cleaning.ipynb`** → `03_labeling_splitting.ipynb`

**Inputs (from Google Drive):**
- `NLP-Complaints-Team/data/text/complaints_labeled.csv`
- `NLP-Complaints-Team/data/text/complaints_unlabeled.csv`

**Outputs (saved to Drive):**
- `NLP-Complaints-Team/data/processed/complaints_labeled_clean.csv`
- `NLP-Complaints-Team/data/processed/complaints_unlabeled_clean.csv`

In [ ]:
from google.colab import drive
import os, re
import pandas as pd

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/NLP-Complaints-Team'
DATA_DIR   = f'{DRIVE_ROOT}/data/text'
PROC_DIR   = f'{DRIVE_ROOT}/data/processed'

print('Drive mounted.')
print('Input dir :', DATA_DIR)
print('Output dir:', PROC_DIR)

In [ ]:
labeled   = pd.read_csv(f'{DATA_DIR}/complaints_labeled.csv',   encoding='utf-8-sig')
unlabeled = pd.read_csv(f'{DATA_DIR}/complaints_unlabeled.csv', encoding='utf-8-sig')

print(f'Labeled   rows: {len(labeled):,}   columns: {labeled.columns.tolist()}')
print(f'Unlabeled rows: {len(unlabeled):,}  columns: {unlabeled.columns.tolist()}')
labeled.head(2)

In [ ]:
def clean_arabic(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub('[\u064B-\u065F\u0670]', '', text)
    text = re.sub('[\u0623\u0625\u0622]', '\u0627', text)
    text = text.replace('\u0629', '\u0647')
    text = text.replace('\u0649', '\u064a')
    text = re.sub('[^\u0600-\u06FF\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(clean_arabic('\u0645\u0631\u062d\u0628\u0627! http://example.com'))

In [ ]:
labeled['text_clean']   = labeled['text'].apply(clean_arabic)
unlabeled['text_clean'] = unlabeled['text'].apply(clean_arabic)

os.makedirs(PROC_DIR, exist_ok=True)

labeled.to_csv(f'{PROC_DIR}/complaints_labeled_clean.csv',   index=False, encoding='utf-8-sig')
unlabeled.to_csv(f'{PROC_DIR}/complaints_unlabeled_clean.csv', index=False, encoding='utf-8-sig')

print(f'Saved complaints_labeled_clean.csv   -> {len(labeled):,} rows')
print(f'Saved complaints_unlabeled_clean.csv -> {len(unlabeled):,} rows')

In [ ]:
pd.set_option('display.max_colwidth', 80)
print('=== Sample: Original vs Cleaned ===')
labeled[['text', 'text_clean']].head(5)